In [11]:
import pandas as pd
import numpy as np
import re

# 1. ĐỌC DỮ LIỆU
df = pd.read_csv('../data/coffee_articles.csv')
print(f"Tổng số bài báo ban đầu: {len(df)}")

# 2. HÀM TRÍCH XUẤT ĐA NĂNG (LẤY CÂU TRÍCH DẪN + LẤY GIÁ)
def extract_quote_and_price(text):
    # Xử lý trường hợp dòng bị rỗng
    if pd.isna(text):
        return pd.Series({'QUOTE': "", 'PRICE': np.nan})
    
    # Tách bài báo thành các câu nhỏ
    sentences = re.split(r'[\n\.]', str(text))
    
    valid_sentences = []
    extracted_prices = []
    
    for sentence in sentences:
        sentence_lower = sentence.lower()
        
        # BỘ LỌC KÉP: Chứa chữ "cà phê" VÀ chứa con số
        if 'cà phê' in sentence_lower and re.search(r'\d+', sentence_lower):
            # Điều kiện phụ: Chứa từ khóa về giá
            if any(keyword in sentence_lower for keyword in ['giá', 'đồng', 'đ/kg', '/kg', 'vnd', 'mức', 'khoảng', 'ở', 'từ']):
                clean_sentence = sentence.strip()
                if clean_sentence:
                    valid_sentences.append(clean_sentence)
                    
                    # --- BÓC TÁCH CON SỐ TỪ CÂU NÀY ---
                    # Kịch bản 1: Viết tắt (VD: "mức 40", "khoảng 42", "thu mua ở 66")
                    matches_short = re.findall(r'(?:mức|khoảng|ở|với|từ|mốc)\s*(\d{2,3})\b', clean_sentence, re.IGNORECASE)
                    for m in matches_short:
                        val = float(m)
                        if 30 <= val <= 150: # Khoảng giá hợp lý 30k - 150k
                            extracted_prices.append(val * 1000) # Quy đổi về chuẩn VND/kg
                            
                    # Kịch bản 2: Viết đầy đủ (VD: "40.000", "65,000", "120.000")
                    matches_full = re.findall(r'(\d{2,3})[\.,](\d{3})\b', clean_sentence)
                    for m1, m2 in matches_full:
                        val = float(m1 + m2)
                        if 30000 <= val <= 150000:
                            extracted_prices.append(val)

    # Gộp các câu trích dẫn tìm được (nếu bài báo nhắc đến giá nhiều lần)
    final_quote = " | ".join(valid_sentences)
    
    # Tính giá trung bình nếu bài báo có báo nhiều mức giá (VD giá ở Đắk Lắk và Gia Lai lệch nhau vài trăm đồng)
    final_price = sum(extracted_prices) / len(extracted_prices) if extracted_prices else np.nan
    
    # Trả về 2 cột dữ liệu
    return pd.Series({'QUOTE': final_quote, 'PRICE': final_price})

# 3. THỰC THI THUẬT TOÁN (Tạo ra 2 cột mới)
print("Đang quét nội dung bài viết và truy xuất nguồn gốc...")
df[['EXTRACTED_QUOTE', 'EXTRACTED_PRICE']] = df['CONTENT'].apply(extract_quote_and_price)

# 4. LÀM SẠCH VÀ SẮP XẾP LẠI BẢNG (Giữ lại các thông tin nguồn)
# Xoá các bài báo không tìm thấy giá (PRICE = NaN)
df_clean = df.dropna(subset=['EXTRACTED_PRICE']).copy()

# Chọn các cột cần thiết theo yêu cầu của bạn
columns_to_keep = ['PUBLISHED_DATE', 'TARGET', 'DOMAIN', 'URL', 'EXTRACTED_QUOTE', 'EXTRACTED_PRICE']
df_paper = df_clean[columns_to_keep].sort_values('PUBLISHED_DATE')

print(f"✅ Đã lọc thành công {len(df_paper)} bài báo có chứa mức giá hợp lệ!")

# 5. XUẤT RA FILE CSV MỚI
df_paper.to_csv('../data/coffee_media_tracker.csv', index=False, encoding='utf-8-sig')

# In thử kết quả (mở rộng cột để đọc được câu trích dẫn và URL)
pd.set_option('display.max_colwidth', 150)
display(df_paper.head())

Tổng số bài báo ban đầu: 9553
Đang quét nội dung bài viết và truy xuất nguồn gốc...
✅ Đã lọc thành công 5410 bài báo có chứa mức giá hợp lệ!


,PUBLISHED_DATE,TARGET,DOMAIN,URL,EXTRACTED_QUOTE,EXTRACTED_PRICE
3775,2021-09-22,arabica,vinanet.vn,https://vinanet.vn/nong-san/gia-ca-phe-hom-nay-229-arabica-duoc-dieu-chinh-tang-sau-chuoi-giam-lien-tiep-749646.html,Giá cà phê hôm nay 22/9: Arabica được điều chỉnh tăng sau chuỗi g | Giá cà phê hôm nay 22/9: Arabica được điều chỉnh tăng sau chuỗi giảm liên tiếp...,39666.666667
3776,2021-09-22,arabica,thuongtruong.com.vn,https://thuongtruong.com.vn/news/gia-ca-phe-hom-nay-229-arabica-tang-tro-lai-66004.html,Giá cà phê hôm nay 22/9: Arabica tăng trở lại | Giá cà phê hôm nay Giá heo hơi hôm nay Giá vàng hôm nay Giá xăng hôm nay Giá tiêu hôm nay Giá ngoạ...,53625.000000
9070,2022-07-24,arabica,kinhtedothi.vn,https://kinhtedothi.vn/gia-ca-phe-hom-nay-24-7-arabica-co-tuan-tang-manh-trong-nuoc-them-1-000-dong-kg,"Giá cà phê hôm nay 24/7: Arabica có tuần tăng mạnh, trong nước thêm 1 | Giá cà phê hôm nay 24/7: Arabica có tuần tăng mạnh, trong nước thêm 1 | Ki...",42090.909091
541,2023-01-08,robusta,baolamdong.vn,https://baolamdong.vn/gia-ca-phe-hom-nay-cap-nhat-17-03-2024-203165.html,Giá cà phê hôm nay – Cập nhật 17/03/2024 | Giá cà phê hôm nay – Cập nhật 17/03/2024 | Giá cà phê hôm nay 17/03/2024 | Giá cà phê hôm nay 17/03 | T...,66714.285714
1080,2023-01-08,arabica,baolamdong.vn,https://baolamdong.vn/gia-ca-phe-hom-nay-cap-nhat-25-12-2023-192214.html,Giá cà phê hôm nay – Cập nhật 25/12/2023 | Giá cà phê hôm nay – Cập nhật 25/12/2023 | Giá cà phê hôm nay 25/12/2023 | Giá cà phê hôm nay 25/12 | T...,66714.285714


In [12]:
arabica_df=pd.read_csv('../data/arabica_vnd.csv')
robusta_df=pd.read_csv('../data/robusta_vnd.csv')

In [13]:
df_paper.iloc[3]

PUBLISHED_DATE                                                                                                                                                2023-01-08
TARGET                                                                                                                                                           robusta
DOMAIN                                                                                                                                                     baolamdong.vn
URL                                                                                             https://baolamdong.vn/gia-ca-phe-hom-nay-cap-nhat-17-03-2024-203165.html
EXTRACTED_QUOTE    Giá cà phê hôm nay – Cập nhật 17/03/2024 | Giá cà phê hôm nay – Cập nhật 17/03/2024 | Giá cà phê hôm nay 17/03/2024 | Giá cà phê hôm nay 17/03 | T...
EXTRACTED_PRICE                                                                                                                                            

In [14]:
df_paper

,PUBLISHED_DATE,TARGET,DOMAIN,URL,EXTRACTED_QUOTE,EXTRACTED_PRICE
3775,2021-09-22,arabica,vinanet.vn,https://vinanet.vn/nong-san/gia-ca-phe-hom-nay-229-arabica-duoc-dieu-chinh-tang-sau-chuoi-giam-lien-tiep-749646.html,Giá cà phê hôm nay 22/9: Arabica được điều chỉnh tăng sau chuỗi g | Giá cà phê hôm nay 22/9: Arabica được điều chỉnh tăng sau chuỗi giảm liên tiếp...,39666.666667
3776,2021-09-22,arabica,thuongtruong.com.vn,https://thuongtruong.com.vn/news/gia-ca-phe-hom-nay-229-arabica-tang-tro-lai-66004.html,Giá cà phê hôm nay 22/9: Arabica tăng trở lại | Giá cà phê hôm nay Giá heo hơi hôm nay Giá vàng hôm nay Giá xăng hôm nay Giá tiêu hôm nay Giá ngoạ...,53625.000000
9070,2022-07-24,arabica,kinhtedothi.vn,https://kinhtedothi.vn/gia-ca-phe-hom-nay-24-7-arabica-co-tuan-tang-manh-trong-nuoc-them-1-000-dong-kg,"Giá cà phê hôm nay 24/7: Arabica có tuần tăng mạnh, trong nước thêm 1 | Giá cà phê hôm nay 24/7: Arabica có tuần tăng mạnh, trong nước thêm 1 | Ki...",42090.909091
541,2023-01-08,robusta,baolamdong.vn,https://baolamdong.vn/gia-ca-phe-hom-nay-cap-nhat-17-03-2024-203165.html,Giá cà phê hôm nay – Cập nhật 17/03/2024 | Giá cà phê hôm nay – Cập nhật 17/03/2024 | Giá cà phê hôm nay 17/03/2024 | Giá cà phê hôm nay 17/03 | T...,66714.285714
1080,2023-01-08,arabica,baolamdong.vn,https://baolamdong.vn/gia-ca-phe-hom-nay-cap-nhat-25-12-2023-192214.html,Giá cà phê hôm nay – Cập nhật 25/12/2023 | Giá cà phê hôm nay – Cập nhật 25/12/2023 | Giá cà phê hôm nay 25/12/2023 | Giá cà phê hôm nay 25/12 | T...,66714.285714
...,...,...,...,...,...,...
9274,NaN,arabica,congthuong.vn,https://congthuong.vn/gia-ca-phe-hom-nay-3062024-gia-ca-phe-trong-nuoc-tiep-da-giam-329112.html,Giá cà phê hôm nay 30/6/2024: Giá cà phê trong nước tiếp đà giảm | Giá cà phê hôm nay 30/6/2024: Giá cà phê trong nước tiếp đà giảm | Cập nhật giá...,104333.333333
9275,NaN,arabica,thuonghieucongluan.com.vn,https://thuonghieucongluan.com.vn/gia-ca-phe-hom-nay-30-6-trong-nuoc-tiep-da-giam-nhe-600-700-dong-kg-a226596.html,Giá cà phê hôm nay 30/6: Trong nước tiếp đà giảm nhẹ 600 -700 đồng/kg | Giá cà phê hôm nay 30/6: Trong nước tiếp đà giảm nhẹ 600 -700 đồng/kg | Gi...,113200.000000
9276,NaN,arabica,trading.vietcap.com.vn,https://trading.vietcap.com.vn/ai-news/post-detail/gia-ca-phe-hom-nay-30-6-gia-on-dinh-du-bao-kho-tang-manh-trong-tuan-dau-thang-7,"Giá cà phê hôm nay 30/6: Giá ổn định, dự báo khó tăng mạnh trong tuần đầu tháng 7 - Chi tiết bài viết | Vietcap AI News | Giá cà phê hôm nay 30/6:...",93500.000000
9316,NaN,arabica,hanghoa.anfin.vn,https://hanghoa.anfin.vn/gia-thi-truong/ca-phe-arabica/,Giá cà phê Arabica hôm nay - Cập nhật giá hợp đồng tương lai 24/7 | Brazil dẫn đầu ngành với sản lượng chiếm khoảng 31 - 38% tổng sản lượng cà phê...,34250.000000
